# Final Assignment: Classifying Premium Home Sales in Ames, Iowa

Supervised Machine Learning: Classification

This notebook builds and compares several classification models, picks the one that best fits the objective, and writes the final report as a PDF (last section). Every number in the conclusions and in the report is computed from the results in this notebook.

## 1. Setup

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, ConfusionMatrixDisplay,
                             RocCurveDisplay, PrecisionRecallDisplay)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
RANDOM_STATE = 42

## 2. Data Description

The data is the Ames Housing dataset (De Cock, 2011), the same one used in the Kaggle "House Prices" competition. The copy used here is `Ames_Housing_Sales.csv` from the course materials. Each row is one home sale in Ames, Iowa, described by 79 features (size, quality and condition of the house, the lot, the garage, the neighborhood, and details of the sale) plus the sale price, `SalePrice`.

In [ ]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML240EN-SkillsNetwork/labs/data/Ames_Housing_Sales.csv"
data = pd.read_csv(URL)

n_rows, n_cols = data.shape
n_num = data.drop(columns="SalePrice").select_dtypes("number").shape[1]
n_cat = data.drop(columns="SalePrice").select_dtypes(exclude="number").shape[1]
n_missing = int(data.isnull().sum().sum())

print(f"rows: {n_rows} | columns: {n_cols} (1 price column + {n_cols - 1} features)")
print(f"numeric features: {n_num} | categorical features: {n_cat}")
print(f"missing values: {n_missing}")
data.head()

### Turning the problem into a classification task

The dataset has no class label, so one has to be defined. I define a **premium home** as a sale in the **top 25% of sale prices** (price at or above the 75th percentile). The label is `1` for premium and `0` otherwise. This gives an imbalanced problem (roughly 1 premium home for every 3 other homes), which is realistic for a screening task.

To avoid leaking the answer, `SalePrice` is removed from the input features. The model has to recognise premium homes using only their characteristics.

In [ ]:
PREMIUM_QUANTILE = 0.75
threshold = data["SalePrice"].quantile(PREMIUM_QUANTILE)

y = (data["SalePrice"] >= threshold).astype(int)
X = data.drop(columns="SalePrice")

print(f"premium threshold: {threshold:,.0f} dollars")
print(y.value_counts().rename({0: "not premium", 1: "premium"}).to_string())

plt.figure(figsize=(7, 3.5))
sns.histplot(data["SalePrice"], bins=40)
plt.axvline(threshold, color="red", ls="--", label=f"premium threshold ({threshold:,.0f})")
plt.legend()
plt.title("Sale price distribution")
plt.show()

## 3. Objective of the Analysis

**Objective.** Build a classifier that identifies premium homes (top 25% by price) from their characteristics, so that an agency or appraiser can screen a large number of listings and focus attention on the homes most likely to be premium. The model is meant for **screening and ranking**, not for predicting an exact price.

**What success looks like.** Both mistakes cost something: missing a premium home (low recall) wastes an opportunity, and flagging ordinary homes as premium (low precision) wastes review effort. Because premium homes are the minority class, accuracy alone would be misleading, since a model that never flags any home would already be about 75% accurate. So the main metrics are **precision, recall, F1, and PR-AUC** on the premium class, with ROC-AUC as a supporting metric.

## 4. Preparing the Data

- 70% train and 30% test, **stratified** so both parts keep the same premium share.
- Numeric features: missing values filled with the median, then standardized.
- Categorical features: missing values filled with the most frequent value, then one-hot encoded (very rare categories are grouped together).

All of this sits inside a `Pipeline`, so every step is fitted only on training data (inside each cross-validation fold too). The test set is touched only once, at the end.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)

print("train:", X_train.shape, "| premium share:", round(y_train.mean(), 3))
print("test :", X_test.shape, "| premium share:", round(y_test.mean(), 3))

num_cols = X_train.select_dtypes("number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), num_cols),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10,
                                               sparse_output=False))]), cat_cols),
])

## 5. Comparing Classification Models

Seven candidates are compared:

1. **Baseline** (always predicts the majority class), as a reference point
2. **Logistic Regression**
3. **K-Nearest Neighbors**
4. **Support Vector Machine** (RBF kernel)
5. **Decision Tree**
6. **Random Forest**
7. **Gradient Boosting**

Each model's hyperparameters are tuned with `GridSearchCV` (5-fold stratified cross validation on the training set only). Models that support it use class weights to deal with the imbalance. The model is selected by its **cross-validated F1 score** for the premium class, not by test results.

In [ ]:
candidates = {
    "Baseline": (DummyClassifier(strategy="prior"), None),
    "Logistic Regression": (
        LogisticRegression(max_iter=5000, class_weight="balanced"),
        {"C": [0.01, 0.1, 1, 10]}),
    "KNN": (
        KNeighborsClassifier(),
        {"n_neighbors": [5, 11, 21], "weights": ["uniform", "distance"]}),
    "SVM (RBF)": (
        SVC(class_weight="balanced"),
        {"C": [0.1, 1, 10]}),
    "Decision Tree": (
        DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        {"max_depth": [3, 5, 8, None], "min_samples_leaf": [1, 5, 10]}),
    "Random Forest": (
        RandomForestClassifier(class_weight="balanced_subsample", random_state=RANDOM_STATE),
        {"n_estimators": [200], "max_depth": [None, 10], "min_samples_leaf": [1, 3]}),
    "Gradient Boosting": (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1], "max_depth": [2, 3]}),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"f1": "f1", "roc_auc": "roc_auc", "pr_auc": "average_precision"}


def get_scores(model, X_):
    # probability of the premium class, or the decision score for models without probabilities
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_)[:, 1]
    return model.decision_function(X_)

In [ ]:
rows, fitted, best_params, test_scores = [], {}, {}, {}

for name, (clf, grid) in candidates.items():
    pipe = Pipeline([("prep", preprocess), ("clf", clf)])

    if grid:
        gs = GridSearchCV(pipe, {f"clf__{k}": v for k, v in grid.items()},
                          cv=cv, scoring=scoring, refit="f1", n_jobs=-1)
        gs.fit(X_train, y_train)
        model, i = gs.best_estimator_, gs.best_index_
        cv_f1 = gs.cv_results_["mean_test_f1"][i]
        cv_f1_std = gs.cv_results_["std_test_f1"][i]
        cv_auc = gs.cv_results_["mean_test_roc_auc"][i]
        best_params[name] = {k.replace("clf__", ""): v for k, v in gs.best_params_.items()}
    else:
        res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
        model = pipe.fit(X_train, y_train)
        cv_f1, cv_f1_std, cv_auc = res["test_f1"].mean(), res["test_f1"].std(), res["test_roc_auc"].mean()
        best_params[name] = {}

    pred_tr, pred_te = model.predict(X_train), model.predict(X_test)
    s_te = get_scores(model, X_test)

    fitted[name], test_scores[name] = model, s_te
    rows.append({
        "model": name,
        "CV F1": cv_f1,
        "CV F1 std": cv_f1_std,
        "CV ROC-AUC": cv_auc,
        "Train F1": f1_score(y_train, pred_tr, zero_division=0),
        "Test Accuracy": accuracy_score(y_test, pred_te),
        "Test Precision": precision_score(y_test, pred_te, zero_division=0),
        "Test Recall": recall_score(y_test, pred_te, zero_division=0),
        "Test F1": f1_score(y_test, pred_te, zero_division=0),
        "Test ROC-AUC": roc_auc_score(y_test, s_te),
        "Test PR-AUC": average_precision_score(y_test, s_te),
    })

results = pd.DataFrame(rows).set_index("model").sort_values("CV F1", ascending=False)
results.round(3)

In [ ]:
print("Selected hyperparameters:")
for name, p in best_params.items():
    print(f"  {name:<20} {p if p else '-'}")

### ROC and precision-recall curves on the test set

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, s in test_scores.items():
    if name == "Baseline":
        continue
    RocCurveDisplay.from_predictions(y_test, s, name=name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_test, s, name=name, ax=axes[1])
axes[0].plot([0, 1], [0, 1], "k:", lw=1)
axes[0].set_title("ROC curves")
axes[1].axhline(y_test.mean(), color="k", ls=":", lw=1)
axes[1].set_title("Precision-recall curves")
plt.tight_layout()
plt.savefig("curves.png", dpi=150)
plt.show()

### Train vs test F1 (checking for overfitting)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
results[["Train F1", "Test F1"]].plot.bar(ax=ax, width=0.75)
ax.set_ylabel("F1 (premium class)")
plt.xticks(rotation=20, ha="right")
fig.tight_layout()
plt.show()

### Selecting the model

In [ ]:
best_name = results["CV F1"].idxmax()
best_model = fitted[best_name]
b = results.loc[best_name]
second = results["CV F1"].sort_values(ascending=False).index[1]

print(f"Selected model (highest cross-validated F1): {best_name}")
print(f"  hyperparameters: {best_params[best_name]}")
print(f"  CV F1 {b['CV F1']:.3f} (+/- {b['CV F1 std']:.3f}) | runner-up: {second} ({results.loc[second, 'CV F1']:.3f})")
print(f"  test precision {b['Test Precision']:.3f} | recall {b['Test Recall']:.3f} | "
      f"F1 {b['Test F1']:.3f} | ROC-AUC {b['Test ROC-AUC']:.3f} | PR-AUC {b['Test PR-AUC']:.3f}")
print(f"  train F1 {b['Train F1']:.3f} vs test F1 {b['Test F1']:.3f} (gap {b['Train F1'] - b['Test F1']:.3f})")
print(f"  baseline accuracy (never flags a premium home): {results.loc['Baseline', 'Test Accuracy']:.3f}")

Why this model: it has the best balance of precision and recall on the premium class in cross validation, which is what the objective asks for. The selection was made without looking at the test set, so the test numbers above are an honest estimate for new data.

## 6. Findings

### 6.1 Where does the selected model make mistakes?

In [ ]:
pred = best_model.predict(X_test)

fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, pred, display_labels=["not premium", "premium"], cmap="Blues", colorbar=False, ax=ax)
ax.set_title(f"{best_name}: confusion matrix (test)")
plt.tight_layout()
plt.savefig("confusion.png", dpi=150)
plt.show()

tn, fp, fn, tp = ((y_test == 0) & (pred == 0)).sum(), ((y_test == 0) & (pred == 1)).sum(), \
                 ((y_test == 1) & (pred == 0)).sum(), ((y_test == 1) & (pred == 1)).sum()
print(f"correctly flagged premium homes: {tp} | missed premium homes: {fn} | "
      f"ordinary homes wrongly flagged: {fp}")

### 6.2 Which features matter most?

I use permutation importance on the test set: shuffle one feature at a time and measure how much the PR-AUC drops. It works for any model and is computed on the original features, so it is easy to read.

In [ ]:
perm = permutation_importance(best_model, X_test, y_test, scoring="average_precision",
                              n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
importance = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)
top_features = importance.head(10)

fig, ax = plt.subplots(figsize=(7, 4))
top_features.sort_values().plot.barh(ax=ax)
ax.set_xlabel("drop in PR-AUC when the feature is shuffled")
ax.set_title(f"Top 10 features ({best_name})")
fig.tight_layout()
fig.savefig("importance.png", dpi=150)
plt.show()

top_features.round(4)

A larger drop means the model relies more on that feature. Features that are strongly correlated with each other can share their importance, so a feature that looks unimportant may simply be overlapping with another one.

### 6.3 Using the model as a screening tool

If the agency can only review a limited share of listings, they would review those with the highest model scores first. This table shows what that would capture on the test set:

In [ ]:
score = pd.Series(test_scores[best_name], index=y_test.index).sort_values(ascending=False)
total_premium = int(y_test.sum())

gain_rows = []
for frac in [0.10, 0.20, 0.25, 0.30, 0.40]:
    k = int(round(frac * len(score)))
    hit = int(y_test.loc[score.index[:k]].sum())
    gain_rows.append({"share of listings reviewed": f"{frac:.0%}",
                      "homes reviewed": k,
                      "premium homes found": hit,
                      "precision": hit / k,
                      "recall": hit / total_premium})
gain = pd.DataFrame(gain_rows).set_index("share of listings reviewed")
gain.round(3)

### 6.4 Summary of findings

In [ ]:
r25 = gain.loc["25%"]
print("KEY FINDINGS")
print("-" * 70)
print(f"1. Best model: {best_name}, chosen by cross-validated F1.")
print(f"2. On unseen test data it reaches precision {b['Test Precision']:.2f}, recall {b['Test Recall']:.2f}, "
      f"F1 {b['Test F1']:.2f}, ROC-AUC {b['Test ROC-AUC']:.2f}.")
print(f"3. It found {tp} of {tp + fn} premium homes in the test set and wrongly flagged {fp} ordinary homes.")
print(f"4. Reviewing the top 25% of listings by score captures {r25['recall']:.0%} of premium homes "
      f"with {r25['precision']:.0%} precision.")
print(f"5. Most influential features: {', '.join(top_features.index[:3])}.")

## 7. Limitations and Next Steps

**Limitations of the model**

- **The label is artificial.** "Premium" is just the top 25% of prices. Homes near the threshold are almost identical to each other, so some errors are unavoidable and the threshold itself is a judgment call.
- **Limited scope.** The data is from one city and a few years. The model may not hold in other markets or when market conditions change.
- **Simple feature handling.** One-hot encoding ignores the natural order of rating-type features (for example quality grades), and rare categories were grouped together.
- **A single train/test split.** The test set is a few hundred homes, so the test metrics have noticeable uncertainty.
- **Importance is not causation.** Permutation importance shows what the model uses, not what drives prices, and correlated features share importance.
- **Class imbalance.** Class weights help but the minority class is still the harder one to learn.

**Plan to improve the analysis**

- *Additional data:* precise location (coordinates, distance to schools and the city center), market conditions at the time of sale (interest rates, price indices), renovation history, and sales from other cities and years.
- *Alternative techniques:* tune gradient boosting libraries (XGBoost, LightGBM) more extensively, try cost-sensitive learning or resampling for the imbalance, encode ordinal features in order, and calibrate predicted probabilities.
- *Better framing:* a multi-class version (low, mid, high, premium), or predict the price with regression and classify afterwards, to compare with this direct approach.
- *Stronger validation:* repeated cross validation, and a time-based split (train on earlier years, test on later years) to simulate real use.

**Next steps**

1. Choose the review threshold based on the real cost of a missed premium home versus a wasted review.
2. Validate the model on newer sales data before relying on it.
3. Add the extra data sources above and re-run this comparison.
4. Monitor performance after deployment and retrain regularly.

## 8. Generate the PDF Report

The cell below writes the final report to `Final_Report_Classification.pdf`, using the numbers computed above (the notebook has to be run from top to bottom first). In Colab it also downloads the file. Set `AUTHOR` if you want a name on the title page.

In [ ]:
import subprocess, sys
try:
    import reportlab
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "reportlab"])

from xml.sax.saxutils import escape
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.utils import ImageReader
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
                                Image, KeepTogether)

AUTHOR = ""   # e.g. "Your Name - Student ID"
REPORT_PATH = "Final_Report_Classification.pdf"

# ---------- styles ----------
ss = getSampleStyleSheet()
NAVY = colors.HexColor("#1f3a5f")
st_title = ParagraphStyle("t", parent=ss["Title"], fontSize=19, leading=23, textColor=NAVY, spaceAfter=4)
st_sub = ParagraphStyle("s", parent=ss["Normal"], fontSize=10, textColor=colors.grey, alignment=1, spaceAfter=10)
st_h1 = ParagraphStyle("h1", parent=ss["Heading1"], fontSize=13.5, textColor=NAVY, spaceBefore=12, spaceAfter=5)
st_h2 = ParagraphStyle("h2", parent=ss["Heading2"], fontSize=11, textColor=NAVY, spaceBefore=8, spaceAfter=3)
st_body = ParagraphStyle("b", parent=ss["BodyText"], fontSize=9.5, leading=13.5, spaceAfter=5)
st_bul = ParagraphStyle("bl", parent=st_body, leftIndent=14, bulletIndent=3, spaceAfter=3)
st_cap = ParagraphStyle("c", parent=st_body, fontSize=8, textColor=colors.grey, alignment=1)


def P(text, style=st_body):
    return Paragraph(text, style)


def bullets(items):
    return [Paragraph(t, st_bul, bulletText="-") for t in items]


def pic(path, width_cm):
    w, h = ImageReader(path).getSize()
    return Image(path, width=width_cm * cm, height=width_cm * cm * h / w)


def table(df, col_widths=None, font=8):
    data = [list(df.columns)] + df.values.tolist()
    t = Table(data, colWidths=col_widths, repeatRows=1)
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), NAVY), ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"), ("FONTSIZE", (0, 0), (-1, -1), font),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#eef2f7")]),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#b8c2cf")),
        ("ALIGN", (1, 0), (-1, -1), "CENTER"), ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("TOPPADDING", (0, 0), (-1, -1), 3), ("BOTTOMPADDING", (0, 0), (-1, -1), 3),
    ]))
    return t


# ---------- numbers used in the text ----------
dummy_acc = results.loc["Baseline", "Test Accuracy"]
gap = b["Train F1"] - b["Test F1"]
gap_text = ("which suggests some overfitting" if gap > 0.10
            else "which suggests the model generalizes reasonably well")
lr_f1 = results.loc["Logistic Regression", "Test F1"]
diff_lr = b["Test F1"] - lr_f1
if best_name == "Logistic Regression":
    lr_text = "The selected model is also one of the simplest candidates, which makes it easy to explain."
elif diff_lr > 0.005:
    lr_text = f"Compared with plain Logistic Regression (test F1 {lr_f1:.3f}), the selected model is {diff_lr:.3f} higher in F1."
elif diff_lr < -0.005:
    lr_text = (f"Plain Logistic Regression reached a slightly higher test F1 ({lr_f1:.3f}), but the selected model "
               f"was better in cross validation, which is the fairer comparison.")
else:
    lr_text = f"Plain Logistic Regression performs almost the same on the test set (F1 {lr_f1:.3f})."

story = []

# ---------- title ----------
story.append(P("Identifying Premium Home Sales in Ames, Iowa", st_title))
story.append(P("A supervised machine learning classification analysis", st_sub))
if AUTHOR:
    story.append(P(escape(AUTHOR), st_sub))

story.append(P("Summary", st_h1))
story.append(P(
    f"This report builds a classifier that recognises <b>premium homes</b> (the top 25% of sale prices) "
    f"from their characteristics. Seven approaches were compared with cross validation and the "
    f"<b>{escape(best_name)}</b> model was selected. On unseen test data it reached precision "
    f"{b['Test Precision']:.2f}, recall {b['Test Recall']:.2f} and F1 {b['Test F1']:.2f}. Used as a screening "
    f"tool, reviewing the top 25% of listings by score finds {r25['recall']:.0%} of premium homes "
    f"with {r25['precision']:.0%} precision."))

# ---------- 1. data description ----------
story.append(P("1. Data Description", st_h1))
story.append(P(
    f"The data is the Ames Housing dataset (De Cock, 2011), also used in the Kaggle House Prices competition. "
    f"It contains <b>{n_rows:,} home sales</b> in Ames, Iowa, described by <b>{n_cols - 1} features</b> "
    f"({n_num} numeric and {n_cat} categorical) and the sale price. The features describe the size, quality and "
    f"condition of the house, the lot, the garage, the neighborhood, and details of the sale (for example "
    f"MSSubClass, MSZoning, SaleType, SaleCondition, MoSold and YrSold). Sale prices range from "
    f"{data['SalePrice'].min():,.0f} to {data['SalePrice'].max():,.0f} dollars with a median of "
    f"{data['SalePrice'].median():,.0f}. There are {n_missing} missing values in the file used."))
story.append(P(
    f"<b>How the features are used.</b> The dataset has no class label, so a home is labelled premium when its "
    f"price is at or above the 75th percentile ({threshold:,.0f} dollars). This gives {int(y.sum()):,} premium "
    f"and {int((1 - y).sum()):,} other homes. The price column is <b>excluded</b> from the inputs so the model "
    f"cannot read the answer directly. All other features are the model inputs: numeric features are "
    f"filled with the median and standardized, categorical features are filled with the most frequent value and "
    f"one-hot encoded, all inside a pipeline fitted on training data only. The data was split 70/30 into train "
    f"({len(X_train):,} homes) and test ({len(X_test):,} homes) sets, stratified by class. After training, "
    f"features are interpreted with permutation importance, which measures how much performance drops when a "
    f"feature is shuffled."))

# ---------- 2. objective ----------
story.append(P("2. Objective of the Analysis", st_h1))
story.append(P(
    "The goal is to identify premium homes from their characteristics so that an agency or appraiser can "
    "screen many listings and focus on those most likely to be premium. The model is intended for "
    "<b>screening and ranking</b>, not for predicting exact prices. Both kinds of mistakes are costly: "
    "missing a premium home wastes an opportunity (low recall), and flagging ordinary homes wastes review "
    "effort (low precision). Since premium homes are the minority, accuracy alone is misleading: a model "
    f"that never flags any home is already {dummy_acc:.0%} accurate. For this reason the main metrics are precision, "
    "recall, F1 and PR-AUC for the premium class, with ROC-AUC as a supporting metric."))

# ---------- 3. model comparison ----------
story.append(P("3. Comparison of Classification Models", st_h1))
story.append(P(
    "Seven candidates were compared: a baseline that always predicts the majority class, Logistic Regression, "
    "K-Nearest Neighbors, a Support Vector Machine (RBF kernel), a Decision Tree, a Random Forest, and Gradient "
    "Boosting. Hyperparameters were tuned with 5-fold stratified cross validation on the training set only, and "
    "the final model was chosen by cross-validated F1 on the premium class. The test set was used once, to "
    "report performance."))

cmp = results.reset_index()[["model", "CV F1", "Test Precision", "Test Recall", "Test F1", "Test ROC-AUC", "Test PR-AUC"]].copy()
cmp.columns = ["Model", "CV F1", "Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]
for c in cmp.columns[1:]:
    cmp[c] = cmp[c].map(lambda v: f"{v:.3f}")
story.append(KeepTogether([
    table(cmp, col_widths=[4.2 * cm] + [2.1 * cm] * 6),
    P("CV F1 is from cross validation on the training set. The other columns are measured on the test set.", st_cap)]))
story.append(Spacer(1, 6))
story.append(pic("curves.png", 16.5))
story.append(P("ROC and precision-recall curves of the tuned models on the test set.", st_cap))
story.append(P(
    f"<b>Best model for the objective: {escape(best_name)}.</b> It had the highest cross-validated F1 "
    f"({b['CV F1']:.3f}), ahead of {escape(second)} ({results.loc[second, 'CV F1']:.3f}). On the test set it "
    f"reached precision {b['Test Precision']:.3f}, recall {b['Test Recall']:.3f}, F1 {b['Test F1']:.3f}, "
    f"ROC-AUC {b['Test ROC-AUC']:.3f} and PR-AUC {b['Test PR-AUC']:.3f}. Its F1 is {b['Train F1']:.3f} on the "
    f"training set and {b['Test F1']:.3f} on the test set (a gap of {gap:.3f}), {gap_text}. {lr_text} "
    f"The baseline has an F1 of 0 because it never flags a premium home, even though its accuracy looks high, "
    f"which is why the evaluation focuses on the premium class."))

# ---------- 4. insights ----------
story.append(P("4. Key Insights and Recommendations", st_h1))
story.append(P("Key insights", st_h2))
story += bullets([
    f"On the test set the model found <b>{tp} of {tp + fn}</b> premium homes and wrongly flagged <b>{fp}</b> "
    f"ordinary homes (out of {tn + fp}).",
    f"The three features the model relies on most are <b>{escape(top_features.index[0])}</b>, "
    f"<b>{escape(top_features.index[1])}</b> and <b>{escape(top_features.index[2])}</b>, based on permutation "
    f"importance.",
    f"As a ranking tool, reviewing the top 25% of listings by score captures <b>{r25['recall']:.0%}</b> of premium "
    f"homes with <b>{r25['precision']:.0%}</b> precision.",
])
story.append(Spacer(1, 4))
fig_row = Table([[pic("confusion.png", 6.6), pic("importance.png", 10.0)],
                 [P("Confusion matrix of the selected model (test set).", st_cap),
                  P("Top 10 features by permutation importance.", st_cap)]],
                colWidths=[7.2 * cm, 10.2 * cm])
fig_row.setStyle(TableStyle([("VALIGN", (0, 0), (-1, 0), "MIDDLE")]))
story.append(KeepTogether([fig_row]))

g = gain.reset_index()
g["precision"] = g["precision"].map(lambda v: f"{v:.0%}")
g["recall"] = g["recall"].map(lambda v: f"{v:.0%}")
g.columns = ["Share of listings reviewed", "Homes reviewed", "Premium found", "Precision", "Recall"]
story.append(KeepTogether([
    P("Screening performance on the test set (listings ranked by model score)", st_h2),
    table(g, col_widths=[4.6 * cm, 3 * cm, 3 * cm, 2.8 * cm, 2.8 * cm])]))

story.append(P("Recommendations", st_h2))
story += bullets([
    "<b>Use the model to rank, then review.</b> Score all listings and send the top-ranked ones for human review. "
    "Pick the review share from the cost of a missed premium home versus a wasted review; the table above shows the trade-off.",
    f"<b>Keep the key inputs accurate.</b> Because the model leans on {escape(top_features.index[0])}, "
    f"{escape(top_features.index[1])} and {escape(top_features.index[2])}, errors in these fields will hurt predictions most, "
    f"so they are the priority for data quality checks.",
    "<b>Do not use it as a price estimator.</b> The model only says whether a home is likely to be in the top quarter. "
    "An exact valuation needs a regression model or an appraisal.",
    "<b>Treat feature importance as a guide, not as cause and effect.</b> It shows what the model uses, not what "
    "makes a home more valuable.",
])

# ---------- 5. limitations & next steps ----------
story.append(P("5. Limitations, Improvement Plan and Next Steps", st_h1))
story.append(P("5.1 Limitations of the model", st_h2))
story += bullets([
    "<b>Artificial label.</b> Premium is just the top 25% of prices. Homes close to the threshold are nearly "
    "identical, so some errors are unavoidable, and the threshold itself is a judgment call.",
    "<b>Limited scope.</b> The data comes from one city over a few years, so the model may not carry over to other "
    "markets or to changing market conditions.",
    "<b>Simple feature handling.</b> One-hot encoding ignores the order of rating-type features, and rare "
    "categories were grouped together.",
    "<b>Single train/test split.</b> The test set has only a few hundred homes, so the test metrics carry noticeable uncertainty.",
    "<b>Importance is not causation,</b> and correlated features share importance.",
    "<b>Class imbalance.</b> Class weights help, but the minority class is still the harder one to learn.",
])
story.append(P("5.2 Plan to improve the analysis", st_h2))
story += bullets([
    "<b>Additional data:</b> precise location (coordinates, distance to schools and the city center), market "
    "conditions at the time of sale (interest rates, price indices), renovation history, and sales from other cities and years.",
    "<b>Alternative techniques:</b> more extensive tuning of gradient boosting libraries (XGBoost, LightGBM), "
    "cost-sensitive learning or resampling for the imbalance, ordinal encoding of rating features, and probability calibration.",
    "<b>Different framing:</b> a multi-class version (low, mid, high, premium), or predicting the price with "
    "regression and then classifying, to compare with the direct approach.",
    "<b>Stronger validation:</b> repeated cross validation and a time-based split (train on earlier years, test on later years).",
])
story.append(P("5.3 Next steps", st_h2))
story += bullets([
    "Choose the review threshold using the real costs of missed and wasted reviews.",
    "Validate the model on newer sales data before relying on it.",
    "Add the extra data sources above and repeat the model comparison.",
    "Monitor performance after deployment and retrain regularly.",
])


def footer(canvas, doc):
    canvas.saveState()
    canvas.setFont("Helvetica", 8)
    canvas.setFillColor(colors.grey)
    canvas.drawCentredString(A4[0] / 2, 1.2 * cm, f"Page {doc.page}")
    canvas.restoreState()


doc = SimpleDocTemplate(REPORT_PATH, pagesize=A4, leftMargin=2 * cm, rightMargin=2 * cm,
                        topMargin=1.8 * cm, bottomMargin=2 * cm,
                        title="Identifying Premium Home Sales in Ames, Iowa", author=AUTHOR or "")
doc.build(story, onFirstPage=footer, onLaterPages=footer)
print("report saved:", REPORT_PATH)

try:
    from google.colab import files
    files.download(REPORT_PATH)
except ImportError:
    pass